# Train your racing agent

You are about to teach a car to drive itself around a city track using
reinforcement learning (PPO). The catch: **you** design the reward function,
the signal that tells the car what "driving well" means. Your trained agent
gets exported as a small JSON file that you upload to the race arena, where
it competes against everyone else's agent on the same track.

Workflow: **Setup** (1 min) -> **Design your reward** -> **Train** (10-30 min)
-> **Export & download** -> hand the file to your instructor.


## 1. Setup

Run this once. It clones the code and installs the RL libraries (~1 minute).


In [ ]:
import os

if os.path.basename(os.getcwd()) != "race-agent-lab":
    if not os.path.isdir("race-agent-lab"):
        !git clone -q https://github.com/sreedath/race-agent-lab.git
    %cd race-agent-lab
%pip install -q gymnasium stable-baselines3
print("setup done")


## 2. How to think about reward design

Every 1/30th of a second, the environment tells you what the car just did,
and your function returns a single number: positive for "good", negative
for "bad". PPO will then find a driving policy that maximises the *sum* of
your rewards over an episode. Three big ideas:

**Dense beats sparse.** A sparse reward ("+100 for finishing a lap, 0
otherwise") is almost unlearnable: the car starts by driving randomly and
will essentially never finish a lap by luck, so it never sees the reward.
A dense reward ("+1 per metre of progress") gives feedback every step and
lets learning bootstrap. Use `sig.delta_s` as your dense backbone.

**Scale matters.** PPO cares about the *relative* size of rewards.
If progress gives ~+0.5 per step and your crash penalty is -0.001, the car
will happily grind along walls, the penalty is noise. If the crash penalty
is -1000, the car learns that *moving at all* is dangerous and parks itself.
Keep terms within a couple orders of magnitude of each other.

**Agents exploit loopholes.** The optimiser maximises what you *wrote*, not
what you *meant*. Classic failure:

```python
def my_reward(sig):
    return sig.speed / 30.0   # "drive fast!"
```

This never mentions *direction*, so a perfectly valid solution is driving
fast in circles, or fast into a wall, scraping along it at full throttle.
Speed is fine as a *bonus*, but progress along the track is what you
actually want. Ask yourself: "what is the laziest way to farm my reward?",
because PPO will find it.

**The signals you get** (fields of `sig`):

| signal | meaning |
|---|---|
| `sig.delta_s` | metres of progress along the track this step (negative = backward) |
| `sig.speed` | speed in m/s, max 30 (negative = reversing) |
| `sig.wall_contact` | True while scraping a wall |
| `sig.new_wall_hit` | True only on the first step of a wall hit |
| `sig.terminated` | True when the episode ends early (stuck / wrong way) |
| `sig.lateral` | metres from the track centre (0 = centre, about 7 = wall) |
| `sig.heading_error` | radians between car heading and track direction |
| `sig.car_contact` | True while touching another car (Level 2 only) |
| `sig.new_car_hit` | True only on the first step of a car-car contact (Level 2 only) |


## 3. Your reward function

Edit this. It is the only code you need to write.


In [ ]:
from racing.env.race_env import RewardSignals


def my_reward(sig: RewardSignals) -> float:
    """Score one control step (1/30 s) of driving. Positive = good."""
    reward = 0.0

    # A dense backbone: reward forward progress along the track.
    reward += sig.delta_s

    # YOUR IDEAS HERE. Some things to consider penalising or rewarding:
    #   - hitting walls (sig.new_wall_hit) vs scraping them (sig.wall_contact)
    #   - getting stuck / going the wrong way (sig.terminated)
    #   - wasting time (a small constant penalty every step)
    #   - staying near the racing line (sig.lateral)... or is the
    #     centerline actually the fastest line through a corner?

    return reward


## 4. Name your agent & choose training length

25k steps: drives, badly. 150k: solid laps. 300k (the cap): competitive.
Longer = better but slower; budget roughly 1-2 minutes per 10k steps on a
free Colab CPU.


In [ ]:
AGENT_NAME = "My Racer"      # shown above your car in the arena (max 24 chars)
TRAINING_STEPS = 150_000     # cap: 300_000

assert AGENT_NAME.strip(), "give your agent a name"
assert 1_000 <= TRAINING_STEPS <= 300_000, "steps must be 1k..300k"


## 5. Train

Watch `ep_rew_mean` (average episode reward) climb. You can press the
**stop button** at any time: the model trained so far is kept and you can
export it right away. Checkpoints are also saved every 25k steps under
`runs/checkpoints/`.


In [ ]:
from racing.train.train_ppo import train

model = train(my_reward, TRAINING_STEPS,
              run_name=AGENT_NAME.strip().replace(" ", "_"))


## 5b. Level 2 (optional): overtaking in traffic

`train(..., n_opponents=3)` puts your car on the track WITH three
slower pre-trained cars. Your LiDAR sees them as obstacles (exactly
like in the arena), and two extra signals come alive:

- `sig.car_contact`: True while touching another car this step
- `sig.new_car_hit`: True only on the first step of a car-car contact

Penalize those, keep rewarding progress, and your agent learns to
PASS cleanly instead of ramming. Wall signals and car signals are
independent, so you can weigh them differently (is shoving another
car worse than brushing a wall? Your call.)

Tips:

- Training in traffic is slower per step; budget accordingly
  (the 300k cap includes these steps).
- A good recipe: get a clean solo driver first, then re-run the
  reward cell with car penalties added and continue with traffic.
- Loophole warning: if car penalties are huge, the laziest strategy
  is to park and let everyone drive away. Keep rewarding progress.
- Export works exactly the same afterwards.


In [ ]:
# OPTIONAL Level 2: first add car penalties to my_reward above
# (e.g. `if sig.new_car_hit: reward -= 5.0`) and re-run that cell,
# then uncomment and run this instead of the solo train() above:

# model = train(my_reward, TRAINING_STEPS, n_opponents=3,
#               run_name=AGENT_NAME.strip().replace(" ", "_"))


## 6. Export & download

Turns your policy into a small JSON file and downloads it. Send this file
to your instructor (or drag it into the race arena's lobby yourself).

To export a *checkpoint* instead of the final model, replace `model` with:
`from stable_baselines3 import PPO; model = PPO.load("runs/checkpoints/<file>.zip", device="cpu")`


In [ ]:
from racing.train.export_policy import export_from_model

path = export_from_model(model, AGENT_NAME)
try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    print(f"not running in Colab; your policy is at: {path}")


## 7. (Optional) Back up to Google Drive

Colab VMs are wiped when the session ends. Run this to keep a copy of your
exported policy and checkpoints in your Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/race-agent-lab"
!cp -v *.json "/content/drive/MyDrive/race-agent-lab/" 2>/dev/null
!cp -rv runs/checkpoints "/content/drive/MyDrive/race-agent-lab/" 2>/dev/null
print("backed up")
